In [1]:
from langchain_community.document_loaders import PyPDFLoader, DirectoryLoader
from langchain_classic.text_splitter import RecursiveCharacterTextSplitter

In [2]:
def load_pdf_files(data):
    loader = DirectoryLoader(
        data,
        glob='*.pdf',
        loader_cls=PyPDFLoader
    )
    documents = loader.load()
    return documents

In [ ]:
extracted = load_pdf_files(r'../data/')

In [4]:
from typing import List
from langchain_core.documents import Document

In [5]:
def filter_to_minimal_docs(docs: List[Document]) -> List[Document]:
    '''
    Given a list of Document objects, return a new list of Document objects
    containing only 'source' in metadata and the original page_content.
    '''
    minimal_docs: List[Document] = []
    for doc in docs:
        src = doc.metadata.get('soruce')
        minimal_docs.append(Document(page_content = doc.page_content, metadata = {'source': src}))

    return minimal_docs

In [6]:
minimal_data = filter_to_minimal_docs(extracted)

In [7]:
minimal_data[555]

Document(metadata={'source': None}, page_content='Purpose\nThe drugs described here are used to treat or prevent\nosteoporosis (brittle bone disease) in women past\nmenopause as well as older men. They also are used pre-\nscribed for Paget’s disease, a painful condition that weak-\nens and deforms bones, and they are used to control calci-\num levels in the blood.\nBone is living tissue. Like other tissue, bone is con-\nstantly being broken down and replaced with new materi-\nal. Normally, there is a balance between the breakdown\nof old bone and its replacement with new bone. But\nwhen something goes wrong with the process, bone dis-\norders may result.\nOsteoporosis is a particular concern for women\nafter menopause, as well as for older men. In osteo-\nporosis, the inside of the bones become porous and thin.\nOver time, this condition weakens the bones and makes\nthem more likely to break. Osteoporosis is four times\nmore common in women than in men. This is because\nwomen have less

In [8]:
def text_split(minimal_data):
    splitter = RecursiveCharacterTextSplitter(
        chunk_size = 1000,
        chunk_overlap = 200,
        length_function = len
    )
    texts_chunks = splitter.split_documents(minimal_data)
    return texts_chunks

In [9]:
text_chunk = text_split(minimal_data)

In [10]:
len(text_chunk)

3428

In [ ]:
from langchain_community.embeddings import HuggingFaceEmbeddings

def download_embeddings():
    '''
    Download and return the HuggingFace Embeddings model.
    '''

    model_name = 'sentence-transformers/all-MiniLM-L6-v2'
    embeddings = HuggingFaceEmbeddings(
        model_name=model_name
    )
    return embeddings

embeddings = download_embeddings()

C:\Users\dell\AppData\Local\Temp\ipykernel_18452\3551343591.py:9: LangChainDeprecationWarning: The class `HuggingFaceEmbeddings` was deprecated in LangChain 0.2.2 and will be removed in 1.0. An updated version of the class exists in the `langchain-huggingface package and should be used instead. To use it run `pip install -U `langchain-huggingface` and import as `from `langchain_huggingface import HuggingFaceEmbeddings``.
  embeddings = HuggingFaceEmbeddings(


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


In [12]:
embeddings.embed_query('hello world')

[-0.034477248787879944,
 0.031023213639855385,
 0.006734968163073063,
 0.02610895223915577,
 -0.03936203941702843,
 -0.16030244529247284,
 0.06692398339509964,
 -0.006441461853682995,
 -0.0474504716694355,
 0.014758830890059471,
 0.07087528705596924,
 0.05552760139107704,
 0.019193364307284355,
 -0.026251312345266342,
 -0.010109573602676392,
 -0.026940463110804558,
 0.022307418286800385,
 -0.02222663164138794,
 -0.149692565202713,
 -0.017493005841970444,
 0.007676235865801573,
 0.05435224995017052,
 0.003254480427131057,
 0.0317259207367897,
 -0.0846213772892952,
 -0.02940598502755165,
 0.051595576107501984,
 0.048124026507139206,
 -0.0033147847279906273,
 -0.05827918276190758,
 0.04196928068995476,
 0.02221066877245903,
 0.1281888484954834,
 -0.02233896031975746,
 -0.011656244285404682,
 0.06292830407619476,
 -0.03287629783153534,
 -0.09122602641582489,
 -0.03117540292441845,
 0.05269957706332207,
 0.04703482240438461,
 -0.0842030718922615,
 -0.03005615808069706,
 -0.02074484899640083

In [13]:
from dotenv import load_dotenv
import os
load_dotenv()

True

In [14]:
PINECONE_API_KEY = os.getenv('PINECONE_API_KEY')
HuggingFace_API_TOKEN = os.getenv('HUGGINGFACE_API_TOKEN')

os.environ['PINECONE_API_KEY'] = PINECONE_API_KEY
os.environ['HUGGINGFACE_API_TOKEN'] = HuggingFace_API_TOKEN

In [15]:
from pinecone import Pinecone

pinecone_api_key = PINECONE_API_KEY

pc = Pinecone(api_key = pinecone_api_key)

In [16]:
from pinecone import ServerlessSpec

index_name = 'medical-chatbot'

if not pc.has_index(index_name):
    pc.create_index(
        name = index_name,
        dimension = 384,
        metric = 'cosine',
        spec = ServerlessSpec(cloud = 'aws', region = 'us-east-1')
    )

index = pc.Index(index_name)

In [18]:
for doc in text_chunk:
    doc.metadata = {k: (v if v is not None else "") for k, v in doc.metadata.items()}

In [19]:
from langchain_pinecone import PineconeVectorStore

docsearch = PineconeVectorStore.from_documents(
    documents = text_chunk,
    embedding = embeddings,
    index_name = index_name
)

In [20]:
## Loading Existing Index

from langchain_pinecone import PineconeVectorStore

docsearch = PineconeVectorStore.from_existing_index(
    index_name = index_name,
    embedding = embeddings
)

In [21]:
retriever = docsearch.as_retriever(search_type = 'similarity', search_kwargs = {'k': 3})

In [24]:
retriever.invoke("What is Acne")

[Document(id='d274b0cd-f3e6-4ab6-b386-15241d66b4f2', metadata={'source': ''}, page_content='The goal of treating moderate acne is to decrease\ninflammation and prevent new comedone formation. One\neffective treatment is topical tretinoin along with a topical\nGALE ENCYCLOPEDIA OF MEDICINE 2 25\nAcne\nAcne vulgaris affecting a woman’s face. Acne is the general\nname given to a skin disorder in which the sebaceous\nglands become inflamed. (Photograph by Biophoto Associ-\nates, Photo Researchers, Inc. Reproduced by permission.)\nGEM - 0001 to 0432 - A  10/22/03 1:41 PM  Page 25'),
 Document(id='cbcafa4d-6df6-42e0-95d5-eca67bccc452', metadata={'source': ''}, page_content='The goal of treating moderate acne is to decrease\ninflammation and prevent new comedone formation. One\neffective treatment is topical tretinoin along with a topical\nGALE ENCYCLOPEDIA OF MEDICINE 2 25\nAcne\nAcne vulgaris affecting a woman’s face. Acne is the general\nname given to a skin disorder in which the sebaceous

In [22]:
from langchain_ollama import ChatOllama

llm = ChatOllama(model = 'qwen2.5-coder:7b')

In [27]:
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser
from langchain_core.runnables import RunnableParallel, RunnablePassthrough

In [28]:
system_prompt = '''You are a assistant for question-answering tasks.
    Use the following retrieved documents to answer the question.
    If you do not know the answer, say you do not know.
    Use three sentences maximum and keep the answer concise.
    \n\n{content}'''

prompt = ChatPromptTemplate.from_messages(
    [
        ('system', system_prompt),
        ('human', '{question}'),
    ]
)

In [29]:
def format_docs(docs):
    return "\n\n".join(doc.page_content for doc in docs)

rag_chain = (
    {
        "context": retriever | format_docs,
        "question": RunnablePassthrough()
    }
    | prompt
    | llm
    | StrOutputParser()
)

# Example usage
# rag_chain.invoke("What is Acne")

In [30]:
rag_chain.invoke("What is Acne")

UnauthorizedException: (401)
Reason: Unauthorized
HTTP response headers: HTTPHeaderDict({'Date': 'Wed, 25 Mar 2026 18:38:22 GMT', 'Content-Type': 'text/plain', 'Content-Length': '12', 'Connection': 'keep-alive', 'x-pinecone-auth-rejected-reason': 'Malformed domain', 'www-authenticate': 'Malformed domain', 'server': 'envoy'})
HTTP response body: Unauthorized
